# Big Five Personality Prediction: LLaMA-3 & Mistral on Pandora

In [60]:

# CELL 1: Install dependencies

!pip install -q transformers accelerate datasets tqdm scikit-learn

In [62]:

# CELL 2: Imports

import torch
import gc
import json
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [63]:
# CELL 3: Configuration

SAMPLE_SIZE = 100

USE_ENRICHED_PROMPT = True

RANDOM_SEED = 42

RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAITS = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

In [64]:

# CELL 4: Load and prepare dataset

print("Loading dataset...")

dataset_hf = load_dataset(
    "Fatima0923/Automated-Personality-Prediction",
    download_mode="force_redownload",
    verification_mode="no_checks"   # avoids metadata issues
)

print(dataset_hf)
print("Columns:", dataset_hf["train"].column_names)


Loading dataset...


train_set.csv: 0.00B [00:00, ?B/s]

val_set.csv: 0.00B [00:00, ?B/s]

eval_set.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16047 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2415 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 16047
    })
    validation: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 2415
    })
    test: Dataset({
        features: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'],
        num_rows: 2415
    })
})
Columns: ['text', 'agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism']


In [65]:
test_df = dataset_hf["test"].to_pandas()
thresholds = {}
print("Thresholds (median splits):")
for trait in ["openness", "conscientiousness", "extraversion",
              "agreeableness", "neuroticism"]:
    median = test_df[trait].median()
    thresholds[trait.capitalize()] = median
    pct = (test_df[trait] > median).mean()
    print(f"  {trait}: median={median:.1f} → {pct*100:.1f}% will be class 1")

def convert_dataset(hf_split, thresholds):
    dataset = []
    for row in hf_split:
        dataset.append({
            "comments": [row["text"]],
            "labels": {
                "Openness": 1 if row["openness"] > thresholds["Openness"] else 0,
                "Conscientiousness": 1 if row["conscientiousness"] > thresholds["Conscientiousness"] else 0,
                "Extraversion": 1 if row["extraversion"] > thresholds["Extraversion"] else 0,
                "Agreeableness": 1 if row["agreeableness"] > thresholds["Agreeableness"] else 0,
                "Neuroticism": 1 if row["neuroticism"] > thresholds["Neuroticism"] else 0,
            }
        })
    return dataset

full_dataset = convert_dataset(dataset_hf["test"], thresholds)

import random
random.seed(RANDOM_SEED)
dataset = random.sample(full_dataset, min(SAMPLE_SIZE, len(full_dataset)))
print(f"\nUsing {len(dataset)} samples")

print("\nClass balance in sample:")
for trait in TRAITS:
    n_pos = sum(1 for d in dataset if d["labels"][trait] == 1)
    print(f"  {trait}: {n_pos}/{len(dataset)} positive ({n_pos/len(dataset)*100:.1f}%)")

Thresholds (median splits):
  openness: median=74.0 → 49.4% will be class 1
  conscientiousness: median=26.0 → 47.1% will be class 1
  extraversion: median=24.0 → 49.6% will be class 1
  agreeableness: median=45.0 → 46.3% will be class 1
  neuroticism: median=48.0 → 49.9% will be class 1

Using 100 samples

Class balance in sample:
  Openness: 61/100 positive (61.0%)
  Conscientiousness: 51/100 positive (51.0%)
  Extraversion: 59/100 positive (59.0%)
  Agreeableness: 52/100 positive (52.0%)
  Neuroticism: 54/100 positive (54.0%)


In [66]:

# CELL 6: Shared helpers

def aggregate_user_comments(user_comments, max_chars=3000):
    text = " ".join(user_comments)
    return text[:max_chars]

def parse_output(output):
    output = output.strip()
    if output.startswith("1"):
        return 1
    elif output.startswith("0"):
        return 0
    return None

trait_descriptions = {
    "Openness": "curiosity, imagination, preference for novelty",
    "Conscientiousness": "organization, discipline, reliability",
    "Extraversion": "sociability, assertiveness, energy",
    "Agreeableness": "empathy, cooperation, kindness",
    "Neuroticism": "emotional instability, anxiety, mood swings"
}

def save_results(results, model_name):
    """Save results to disk so they survive Colab disconnects."""
    path = os.path.join(RESULTS_DIR, f"{model_name}_results.json")
    serializable = {k: {"y_true": v[0], "y_pred": v[1]} for k, v in results.items()}
    with open(path, "w") as f:
        json.dump(serializable, f)
    print(f"Results saved to {path}")

def print_summary(results, model_label):
    print(f"\n{'='*50}")
    print(f"SUMMARY: {model_label}")
    print(f"{'='*50}")
    for trait in TRAITS:
        if trait not in results:
            continue
        y_true, y_pred = results[trait]
        print(f"\n--- {trait} ---")
        print(classification_report(y_true, y_pred, digits=3))
        print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
        print(f"F1 (macro): {f1_score(y_true, y_pred, average='macro'):.3f}")

---
# Part 1: LLaMA-3-8B-Instruct

In [67]:

# CELL 7: Load LLaMA

llama_model_name = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading LLaMA tokenizer...")
llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)

print("Loading LLaMA model (this may take 3-5 minutes)...")
llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
llama_model.eval()
print("LLaMA loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading LLaMA tokenizer...
Loading LLaMA model (this may take 3-5 minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LLaMA loaded.
VRAM used: 7.43 GB


In [69]:

# CELL 8: LLaMA prompt builders

positive_facets = {
    "Openness": "imaginative, intellectually curious, appreciates art, open to new ideas",
    "Conscientiousness": "efficient, organized, reliable, plans ahead, follows through",
    "Extraversion": "assertive, energetic, enjoys social interaction, expressive",
    "Agreeableness": "compassionate, cooperative, trusting, helpful",
    "Neuroticism": "experiences anxiety, mood swings, emotional instability"
}

negative_facets = {
    "Openness": "prefers routine, lacks curiosity, avoids new ideas",
    "Conscientiousness": "careless, disorganized, unreliable",
    "Extraversion": "introverted, quiet, avoids social interaction",
    "Agreeableness": "antagonistic, critical, uncooperative",
    "Neuroticism": "emotionally stable, calm, not easily upset"
}

positive_adjectives = {
    "Openness": "creative, imaginative, curious",
    "Conscientiousness": "organized, disciplined, dependable",
    "Extraversion": "outgoing, lively, talkative",
    "Agreeableness": "kind, warm, considerate",
    "Neuroticism": "anxious, sensitive, tense"
}

negative_adjectives = {
    "Openness": "unimaginative, conventional",
    "Conscientiousness": "lazy, careless",
    "Extraversion": "reserved, quiet",
    "Agreeableness": "harsh, rude",
    "Neuroticism": "calm, emotionally stable"
}


def build_llama_simple_prompt(text, trait):
    return (
        f'You are a psychological text classifier.\n\n'
        f'Your task is to determine whether the Big Five trait "{trait}" is expressed in the TEXT.\n\n'
        f'Rules:\n'
        f'- Output ONLY "1" if the trait is present\n'
        f'- Output ONLY "0" if the trait is not present\n'
        f'- Do NOT explain\n'
        f'- Do NOT output anything else\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )

def build_llama_enriched_prompt(text, trait):
    return (
        f"You are an expert in Automatic Personality Prediction using the Big Five model.\n\n"

        f"Task: Determine whether the trait '{trait}' is present in the TEXT.\n\n"

        f"### Trait Overview\n"
        f"{trait_descriptions[trait]}\n\n"

        f"### Positive Indicators (predict 1)\n"
        f"{positive_facets[trait]}\n\n"

        f"### Negative Indicators (predict 0)\n"
        f"{negative_facets[trait]}\n\n"

        f"### Adjectives\n"
        f"High {trait}: {positive_adjectives[trait]}\n"
        f"Low {trait}: {negative_adjectives[trait]}\n\n"

        f"Guidelines:\n"
        f"- Use tone, behavior, emotional cues\n"
        f"- Consider implicit meaning\n\n"

        f"Output ONLY one digit: 1 or 0.\n\n"

        f"TEXT:\n{text}\n\nAnswer:"
    )

In [16]:

# CELL 9: LLaMA inference

@torch.no_grad()
def predict_llama(text, trait, enriched=True):
    prompt = build_llama_enriched_prompt(text, trait) if enriched else build_llama_simple_prompt(text, trait)

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(llama_model.device)

    # Save this BEFORE calling generate
    input_length = inputs["input_ids"].shape[-1]

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id
    )

    # Use saved length instead of re-accessing inputs
    new_tokens = outputs[0][input_length:]
    decoded = llama_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_output(decoded)

In [17]:

# CELL 10: Run LLaMA evaluation

llama_results = {}
invalid_counts = {}

for trait in TRAITS:
    y_true = []
    y_pred = []
    n_invalid = 0

    print(f"\nEvaluating {trait}...")
    for user in tqdm(dataset, desc=trait):
        text = aggregate_user_comments(user["comments"])
        pred = predict_llama(text, trait, enriched=USE_ENRICHED_PROMPT)

        if pred is None:
            n_invalid += 1
            continue

        y_true.append(user["labels"][trait])
        y_pred.append(pred)

    llama_results[trait] = (y_true, y_pred)
    invalid_counts[trait] = n_invalid
    print(f"  Invalid outputs: {n_invalid}/{len(dataset)} ({n_invalid/len(dataset)*100:.1f}%)")
    print(classification_report(y_true, y_pred, digits=3))

    save_results(llama_results, "llama")

print_summary(llama_results, "LLaMA-3-8B-Instruct")


Evaluating Openness...


Openness: 100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


  Invalid outputs: 1/100 (1.0%)
              precision    recall  f1-score   support

           0      0.362     0.641     0.463        39
           1      0.533     0.267     0.356        60

    accuracy                          0.414        99
   macro avg      0.448     0.454     0.409        99
weighted avg      0.466     0.414     0.398        99

Results saved to /content/results/llama_results.json

Evaluating Conscientiousness...


Conscientiousness: 100%|██████████| 100/100 [00:47<00:00,  2.10it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

  Invalid outputs: 1/100 (1.0%)
              precision    recall  f1-score   support

           0      0.485     1.000     0.653        48
           1      0.000     0.000     0.000        51

    accuracy                          0.485        99
   macro avg      0.242     0.500     0.327        99
weighted avg      0.235     0.485     0.317        99

Results saved to /content/results/llama_results.json

Evaluating Extraversion...


...

Agreeableness: 100%|██████████| 100/100 [00:47<00:00,  2.09it/s]


  Invalid outputs: 1/100 (1.0%)
              precision    recall  f1-score   support

           0      0.482     0.833     0.611        48
           1      0.500     0.157     0.239        51

    accuracy                          0.485        99
   macro avg      0.491     0.495     0.425        99
weighted avg      0.491     0.485     0.419        99

Results saved to /content/results/llama_results.json

Evaluating Neuroticism...


Neuroticism: 100%|██████████| 100/100 [00:47<00:00,  2.09it/s]

  Invalid outputs: 1/100 (1.0%)
              precision    recall  f1-score   support

           0      0.493     0.733     0.589        45
           1      0.625     0.370     0.465        54

    accuracy                          0.535        99
   macro avg      0.559     0.552     0.527        99
weighted avg      0.565     0.535     0.522        99

Results saved to /content/results/llama_results.json

SUMMARY: LLaMA-3-8B-Instruct

--- Openness ---
              precision    recall  f1-score   support

           0      0.362     0.641     0.463        39
           1      0.533     0.267     0.356        60

    accuracy                          0.414        99
   macro avg      0.448     0.454     0.409        99
weighted avg      0.466     0.414     0.398        99

Accuracy: 0.414
F1 (macro): 0.409

--- Conscientiousness ---
              precision    recall  f1-score   support

           0      0.485     1.000     0.653        48
           1      0.000     0.000     0.000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:

# CELL 11: Free LLaMA from VRAM before loading Mistral

print(f"VRAM before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

del llama_model
del llama_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("LLaMA freed. Ready to load Mistral.")

VRAM before cleanup: 14.32 GB
VRAM after cleanup: 7.17 GB
LLaMA freed. Ready to load Mistral.


---
# Part 2: Mistral-7B-Instruct

In [70]:

# CELL 12: Load Mistral

mistral_model_name = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading Mistral tokenizer...")
mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_name)

print("Loading Mistral model (3-5 minutes)...")
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
mistral_model.eval()
print("Mistral loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading Mistral tokenizer...
Loading Mistral model (3-5 minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Mistral loaded.
VRAM used: 14.68 GB


In [72]:

# CELL 13: Mistral prompt builders (chat-template format)

def build_mistral_simple_prompt(text, trait):
    return [{
        "role": "user",
        "content": (
            f'You are a psychological text classifier.\n\n'
            f'Determine whether the Big Five trait "{trait}" is expressed in the TEXT.\n\n'
            f'Output ONLY "1" if present, "0" if not. No explanation.\n\n'
            f'TEXT:\n{text}'
        )
    }]

def build_mistral_enriched_prompt(text, trait):
    return [{
        "role": "user",
        "content": (
            f'You are an expert in personality psychology using the Big Five model.\n\n'
            f'Determine whether the trait "{trait}" is present in the TEXT.\n\n'
            f'Trait definition: {trait_descriptions[trait]}\n\n'
            f'Guidelines: Look at tone, wording, emotional expression, and behavior. '
            f'Consider both explicit and implicit signals.\n\n'
            f'Output ONLY one digit: "1" if present, "0" if absent.\n\n'
            f'TEXT:\n{text}'
        )
    }]

In [73]:

# CELL 14: Mistral inference 

import re

def parse_prediction(decoded_text):
    # Example: extract first number 0–1 or 1–5 etc.
    match = re.search(r"([0-9]*\.?[0-9]+)", decoded_text)
    
    if match:
        return float(match.group(1))
    
    return None

@torch.no_grad()
def predict_mistral(text, trait, enriched=False):
    # 1. Build text prompt (NOT chat dict)
    base_prompt = (
        build_mistral_enriched_prompt(text, trait)
        if enriched
        else build_mistral_enriched_prompt(text, trait)
    )

    # 2. Wrap in Mistral instruction format
    prompt = f"<s>[INST] {base_prompt} [/INST]"

    # 3. Tokenize → ALWAYS returns tensors
    encoded = mistral_tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,   # IMPORTANT: disable dynamic padding here
        truncation=True
    )

    input_ids = encoded["input_ids"].to(mistral_model.device)
    attention_mask = encoded["attention_mask"].to(mistral_model.device)

    # 4. Generate
    outputs = mistral_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=10,
        do_sample=False
    )

    # 5. Decode
    decoded = mistral_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return parse_prediction(decoded)

In [45]:
print("PAD TOKEN:", mistral_tokenizer.pad_token)
print("EOS TOKEN:", mistral_tokenizer.eos_token)
print("PAD TOKEN ID:", mistral_tokenizer.pad_token_id)

PAD TOKEN: None
EOS TOKEN: </s>
PAD TOKEN ID: None


In [46]:

# CELL 15: Run Mistral evaluation

mistral_results = {}
mistral_invalid_counts = {}

for trait in TRAITS:
    y_true = []
    y_pred = []
    n_invalid = 0

    print(f"\nEvaluating {trait}...")
    for user in tqdm(dataset, desc=trait):
        text = aggregate_user_comments(user["comments"])
        pred = predict_mistral(text, trait)

        if pred is None:
            n_invalid += 1
            continue

        y_true.append(user["labels"][trait])
        y_pred.append(pred)

    mistral_results[trait] = (y_true, y_pred)
    mistral_invalid_counts[trait] = n_invalid
    print(f"  Invalid outputs: {n_invalid}/{len(dataset)} ({n_invalid/len(dataset)*100:.1f}%)")
    print(classification_report(y_true, y_pred, digits=3))

    save_results(mistral_results, "mistral")

print_summary(mistral_results, "Mistral-7B-Instruct-v0.3")


Evaluating Openness...


Openness: 100%|██████████| 100/100 [10:42<00:00,  6.43s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

  Invalid outputs: 0/100 (0.0%)
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        39
           1      0.610     1.000     0.758        61

    accuracy                          0.610       100
   macro avg      0.305     0.500     0.379       100
weighted avg      0.372     0.610     0.462       100

Results saved to /content/results/mistral_results.json

Evaluating Conscientiousness...


Conscientiousness: 100%|██████████| 100/100 [10:59<00:00,  6.59s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

  Invalid outputs: 0/100 (0.0%)
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        49
           1      0.510     1.000     0.675        51

    accuracy                          0.510       100
   macro avg      0.255     0.500     0.338       100
weighted avg      0.260     0.510     0.345       100

Results saved to /content/results/mistral_results.json

Evaluating Extraversion...


...

Agreeableness: 100%|██████████| 100/100 [10:53<00:00,  6.53s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

  Invalid outputs: 0/100 (0.0%)
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        48
           1      0.520     1.000     0.684        52

    accuracy                          0.520       100
   macro avg      0.260     0.500     0.342       100
weighted avg      0.270     0.520     0.356       100

Results saved to /content/results/mistral_results.json

Evaluating Neuroticism...


Neuroticism: 100%|██████████| 100/100 [10:38<00:00,  6.39s/it]

  Invalid outputs: 0/100 (0.0%)
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        46
           1      0.540     1.000     0.701        54

    accuracy                          0.540       100
   macro avg      0.270     0.500     0.351       100
weighted avg      0.292     0.540     0.379       100

Results saved to /content/results/mistral_results.json

SUMMARY: Mistral-7B-Instruct-v0.3

--- Openness ---
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        39
           1      0.610     1.000     0.758        61

    accuracy                          0.610       100
   macro avg      0.305     0.500     0.379       100
weighted avg      0.372     0.610     0.462       100

Accuracy: 0.610
F1 (macro): 0.379

--- Conscientiousness ---
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        49
           1      0.510     1.000   


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/

In [47]:
import json
with open("/content/results/mistral_results.json", "r") as f:
    saved = json.load(f)
print("Saved traits:", list(saved.keys()))

Saved traits: ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']


In [48]:

# CELL 16: Side-by-side comparison table

print("\n" + "="*60)
print("COMPARISON TABLE: LLaMA vs Mistral (Enriched Prompt, Pandora)")
print("="*60)
print(f"{'Trait':<20} {'LLaMA Acc':>10} {'LLaMA F1':>10} {'Mistral Acc':>12} {'Mistral F1':>10}")
print("-"*60)

for trait in TRAITS:
    la, lp = llama_results.get(trait, ([], []))
    ma, mp = mistral_results.get(trait, ([], []))

    l_acc = accuracy_score(la, lp) if la else float('nan')
    l_f1  = f1_score(la, lp, average='macro') if la else float('nan')
    m_acc = accuracy_score(ma, mp) if ma else float('nan')
    m_f1  = f1_score(ma, mp, average='macro') if ma else float('nan')

    print(f"{trait:<20} {l_acc:>10.3f} {l_f1:>10.3f} {m_acc:>12.3f} {m_f1:>10.3f}")

print("\nPre-LLM SOTA on Pandora (Di Cursi et al. 2025): ~0.50-0.55 macro-F1 (no config consistently exceeds)")
print("Results saved to:", RESULTS_DIR)


COMPARISON TABLE: LLaMA vs Mistral (Enriched Prompt, Pandora)
Trait                 LLaMA Acc   LLaMA F1  Mistral Acc Mistral F1
------------------------------------------------------------
Openness                  0.414      0.409        0.610      0.379
Conscientiousness         0.485      0.327        0.510      0.338
Extraversion              0.505      0.455        0.590      0.371
Agreeableness             0.485      0.425        0.520      0.342
Neuroticism               0.535      0.527        0.540      0.351

Pre-LLM SOTA on Pandora (Di Cursi et al. 2025): ~0.50-0.55 macro-F1 (no config consistently exceeds)
Results saved to: /content/results


# Predicting Audience Personality from Text Testing 

In [74]:

# High-level trait descriptions (Costa & McCrae 1992)
trait_overview = {
    "Openness": (
        "High-Openness individuals are imaginative and sensitive to art and beauty. "
        "They prefer variety and are intellectually curious, creative, and open to new experiences. "
        "Low-Openness individuals are conventional, prefer routine, and tend toward practical thinking."
    ),
    "Conscientiousness": (
        "Conscientiousness reflects organization, self-discipline, and goal-directed behavior. "
        "High scorers are reliable, punctual, and thorough. "
        "Low scorers tend to be disorganized, impulsive, and careless with obligations."
    ),
    "Extraversion": (
        "Extraversion reflects sociability, assertiveness, and positive emotionality. "
        "High scorers are energetic, talkative, and seek social stimulation. "
        "Low scorers (introverts) are reserved, prefer solitude, and find social interaction draining."
    ),
    "Agreeableness": (
        "Agreeableness is primarily a dimension of interpersonal behavior. "
        "High scorers are empathetic, cooperative, and warm toward others. "
        "Low scorers are competitive, skeptical, and may be perceived as cold or antagonistic."
    ),
    "Neuroticism": (
        "Neuroticism reflects emotional instability and vulnerability to stress. "
        "High scorers experience frequent anxiety, moodiness, and emotional distress. "
        "Low scorers are calm, emotionally stable, and resilient under pressure."
    ),
}

# Facet-level cues from IPIP inventory (Di Cursi et al. Appendix D)
ipip_facets = {
    "Openness": {
        "positive": [
            "Has a rich vocabulary and vivid imagination.",
            "Spends time reflecting on things and is full of ideas.",
            "Loves to read challenging material and think up new ways of doing things.",
            "Carries conversations to a higher level.",
            "Is quick to understand abstract concepts.",
        ],
        "negative": [
            "Has difficulty understanding abstract ideas.",
            "Is not interested in abstract ideas or art.",
            "Avoids difficult reading material.",
            "Does not have a good imagination.",
            "Will not probe deeply into a subject.",
        ],
    },
    "Conscientiousness": {
        "positive": [
            "Is always prepared and pays attention to details.",
            "Gets chores done right away and follows a schedule.",
            "Makes plans and sticks to them.",
            "Is exacting in their work and continues until everything is perfect.",
        ],
        "negative": [
            "Leaves belongings around and makes a mess of things.",
            "Shirks duties and wastes time.",
            "Does things in a half-way manner.",
            "Finds it difficult to get down to work.",
        ],
    },
    "Extraversion": {
        "positive": [
            "Is the life of the party and feels comfortable around people.",
            "Starts conversations and talks to many different people.",
            "Does not mind being the center of attention.",
            "Is skilled in handling social situations.",
        ],
        "negative": [
            "Does not talk a lot and keeps in the background.",
            "Finds it difficult to approach others.",
            "Is quiet around strangers and very private.",
            "Often feels uncomfortable around others.",
        ],
    },
    "Agreeableness": {
        "positive": [
            "Is interested in people and sympathizes with others' feelings.",
            "Takes time out for others and makes people feel at ease.",
            "Has a good word for everyone and shows gratitude.",
            "Thinks of others first and loves to help.",
        ],
        "negative": [
            "Insults people and is not interested in others' problems.",
            "Feels little concern for others and is hard to get to know.",
            "Is indifferent to the feelings of others.",
        ],
    },
    "Neuroticism": {
        "positive": [  # High neuroticism indicators
            "Gets stressed out easily and worries about things.",
            "Is easily disturbed and gets upset easily.",
            "Changes mood a lot and has frequent mood swings.",
            "Often feels blue and panics easily.",
            "Gets overwhelmed by emotions and takes offense easily.",
        ],
        "negative": [  # Low neuroticism indicators
            "Is relaxed most of the time and seldom feels blue.",
            "Is not easily bothered by things.",
            "Rarely gets irritated or mad.",
        ],
    },
}

# Goldberg (1990) adjective lists
goldberg_adjectives = {
    "Openness": {
        "positive": ["intelligent", "philosophical", "complex", "deep", "insightful",
                     "creative", "curious", "perceptive", "intellectual", "imaginative",
                     "literary", "artistic", "thoughtful", "meditative"],
        "negative": ["simple", "ignorant", "dull", "illogical", "narrow"],
    },
    "Conscientiousness": {
        "positive": ["organized", "persistent", "thorough", "orderly", "dependable",
                     "punctual", "ambitious", "disciplined", "predictable", "conscientious"],
        "negative": ["messy", "forgetful", "lazy", "careless", "erratic",
                     "absent-minded", "impulsive", "disorganized"],
    },
    "Extraversion": {
        "positive": ["jolly", "talkative", "lively", "social", "outgoing", "assertive",
                     "energetic", "playful", "companionable", "cheerful"],
        "negative": ["reserved", "quiet", "withdrawn", "introverted", "aloof",
                     "bashful", "shy", "modest", "distant"],
    },
    "Agreeableness": {
        "positive": ["friendly", "generous", "kind", "warm", "cooperative", "patient",
                     "tactful", "helpful", "sensitive", "loyal", "trustful", "moral"],
        "negative": ["critical", "harsh", "stubborn", "argumentative", "suspicious",
                     "selfish", "sarcastic", "aggressive", "jealous", "cold"],
    },
    "Neuroticism": {
        "positive": ["touchy", "nervous", "fearful", "unstable", "oversensitive",
                     "whiny", "moody", "self-critical", "timid", "dependent"],
        "negative": ["calm", "stable", "confident", "peaceful", "resilient",
                     "tough", "resourceful", "unflappable"],
    },
}


def build_simple_prompt(text, trait):
    """Minimal prompt — replicates Di Cursi et al. simple condition."""
    return (
        f'You are a psychological text classifier. '
        f'Your job is to analyze a written text and decide if the '
        f'Big Five trait **{trait}** is expressed. '
        f'You are evaluating the content of the text, not the person. '
        f'If the trait is present in the text, respond with 1. '
        f'If not, respond with 0. '
        f'You must always respond with a single digit: 0 or 1. '
        f'Do not explain or add anything else.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )

def build_enriched_prompt(text, trait):
    """
    Enriched prompt — replicates Di Cursi et al. complex condition.
    Combines trait overview + IPIP facets + Goldberg adjectives.
    """
    pos_facets = "\n".join(f"- {f}" for f in ipip_facets[trait]["positive"])
    neg_facets = "\n".join(f"- {f}" for f in ipip_facets[trait]["negative"])
    pos_adj = ", ".join(goldberg_adjectives[trait]["positive"])
    neg_adj = ", ".join(goldberg_adjectives[trait]["negative"])

    return (
        f'You are an expert in Automatic Personality Prediction '
        f'using the Five Factor Model (Big Five / OCEAN).\n\n'
        f'Estimate whether the trait **{trait}** is reflected in the style '
        f'of the text, where "1" means present and "0" means absent.\n\n'
        f'Base your judgment on both explicit and implicit cues — '
        f'including tone, content, emotional framing, and vocabulary.\n\n'
        f'---\n'
        f'### Trait Overview\n'
        f'{trait_overview[trait]}\n\n'
        f'---\n'
        f'### Facet-Level Insight (from IPIP personality inventory)\n'
        f'If the text reflects these → output "1":\n{pos_facets}\n\n'
        f'If the text reflects these → output "0":\n{neg_facets}\n\n'
        f'---\n'
        f'### Adjective Insight (Goldberg 1990)\n'
        f'Words associated with output "1": {pos_adj}\n'
        f'Words associated with output "0": {neg_adj}\n\n'
        f'---\n'
        f'Respond with ONLY a single digit: 0 or 1.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )


def build_audience_prompt(text, trait):
    """
    Audience inference prompt — the novel framing of this paper.
    
    Unlike standard APPT prompts (Di Cursi et al., Gjurkovic et al.)
    which ask 'does the AUTHOR exhibit trait X?', this prompt asks
    'would a person high in trait X be drawn to this text as a reader?'
    
    This operationalizes psychographic audience inference:
    given a source text, predict the personality profile of its
    likely audience — the core gap identified in this paper.
    """
    pos_facets = "\n".join(f"- {f}" for f in ipip_facets[trait]["positive"])
    neg_facets = "\n".join(f"- {f}" for f in ipip_facets[trait]["negative"])
    pos_adj = ", ".join(goldberg_adjectives[trait]["positive"])
    neg_adj = ", ".join(goldberg_adjectives[trait]["negative"])

    return (
        f'You are an expert in psychographic audience analysis '
        f'using the Five Factor Model (Big Five / OCEAN).\n\n'
        f'Your task is NOT to assess the personality of the author.\n'
        f'Instead, determine whether a person HIGH in the trait **{trait}** '
        f'would be drawn to READ, ENGAGE WITH, or SHARE this text.\n\n'
        f'Consider: Does the tone, topic, vocabulary, and emotional register '
        f'of this text resonate with someone who exhibits high {trait}?\n\n'
        f'---\n'
        f'### What high {trait} looks like in a reader\n'
        f'{trait_overview[trait]}\n\n'
        f'---\n'
        f'### Facet-Level Signals — a high-{trait} reader tends to:\n'
        f'{pos_facets}\n\n'
        f'### A low-{trait} reader (who would NOT be drawn to this) tends to:\n'
        f'{neg_facets}\n\n'
        f'---\n'
        f'### Adjective profile of a high-{trait} audience\n'
        f'They are typically: {pos_adj}\n'
        f'They are NOT typically: {neg_adj}\n\n'
        f'---\n'
        f'Output "1" if a high-{trait} person would be drawn to this text.\n'
        f'Output "0" if they would not.\n'
        f'Respond with ONLY a single digit: 0 or 1.\n\n'
        f'TEXT:\n{text}\n\nAnswer:'
    )


@torch.no_grad()
def predict_llama(text, trait, prompt_type="enriched"):
    if prompt_type == "simple":
        prompt = build_simple_prompt(text, trait)
    elif prompt_type == "enriched":
        prompt = build_enriched_prompt(text, trait)
    elif prompt_type == "audience":
        prompt = build_audience_prompt(text, trait)
    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}")

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(llama_model.device)

    input_length = inputs["input_ids"].shape[-1]

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id
    )

    new_tokens = outputs[0][input_length:]
    decoded = llama_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_output(decoded)


@torch.no_grad()
def predict_mistral(text, trait, prompt_type="enriched"):
    if prompt_type == "simple":
        content = build_simple_prompt(text, trait)
    elif prompt_type == "enriched":
        content = build_enriched_prompt(text, trait)
    elif prompt_type == "audience":
        content = build_audience_prompt(text, trait)
    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}")

    messages = [{"role": "user", "content": content}]

    encoded = mistral_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        truncation=True,
        max_length=512
    ).to(mistral_model.device)

    input_length = encoded.shape[-1]

    outputs = mistral_model.generate(
        encoded,
        max_new_tokens=5,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=mistral_tokenizer.eos_token_id
    )

    new_tokens = outputs[0][input_length:]
    decoded = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_output(decoded)

In [ ]:
for prompt_type in ["simple", "enriched", "audience"]:
    print(f"\n{'='*50}")
    print(f"CONDITION: {prompt_type.upper()} PROMPT")
    print(f"{'='*50}")
    results = {}
    for trait in TRAITS:
        y_true, y_pred = [], []
        for user in tqdm(dataset, desc=trait):
            text = aggregate_user_comments(user["comments"])
            pred = predict_llama(text, trait, prompt_type=prompt_type)
            if pred is not None:
                y_true.append(user["labels"][trait])
                y_pred.append(pred)
        results[trait] = (y_true, y_pred)
        print(f"\n{trait}:")
        print(classification_report(y_true, y_pred, digits=3))
    save_results(results, f"llama_{prompt_type}")